# Sequence Encoding for Proteins

This notebook covers different methods to encode protein sequences numerically.

**Learning Objectives:**
- Implement one-hot, BLOSUM, and physicochemical encodings
- Compare encoding properties
- Prepare sequence data for neural networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

## 1. Standard Amino Acids

In [ ]:
# The 20 standard amino acids
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

print("Amino Acid Index Mapping:")
for aa, idx in AA_TO_IDX.items():
    print(f"  {aa}: {idx}")

In [ ]:
# Sample sequences
ubiquitin = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
crambin = "TTCCPSIVARSNFNVCRLPGTPEAICATYTGCIIIPGATCPGDYAN"

print(f"Ubiquitin: {len(ubiquitin)} residues")
print(f"Crambin: {len(crambin)} residues")

## 2. One-Hot Encoding

In [ ]:
def one_hot_encode(sequence, include_unknown=True):
    """
    One-hot encode a protein sequence.
    
    Args:
        sequence: Amino acid sequence string
        include_unknown: Add extra dimension for unknown residues
    
    Returns:
        (L, 20) or (L, 21) numpy array
    """
    n_classes = 21 if include_unknown else 20
    encoding = np.zeros((len(sequence), n_classes), dtype=np.float32)
    
    for i, aa in enumerate(sequence.upper()):
        if aa in AA_TO_IDX:
            encoding[i, AA_TO_IDX[aa]] = 1.0
        elif include_unknown:
            encoding[i, 20] = 1.0  # Unknown token
    
    return encoding

# Encode ubiquitin
ubq_onehot = one_hot_encode(ubiquitin)
print(f"Shape: {ubq_onehot.shape}")
print(f"\nFirst 5 residues ({ubiquitin[:5]}):")
print(ubq_onehot[:5])

In [ ]:
# Visualize one-hot encoding
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(ubq_onehot[:30].T, aspect='auto', cmap='Blues')
ax.set_yticks(range(20))
ax.set_yticklabels(list(AMINO_ACIDS))
ax.set_xlabel('Position')
ax.set_ylabel('Amino Acid')
ax.set_title('One-Hot Encoding (First 30 residues of Ubiquitin)')

# Add sequence below
for i, aa in enumerate(ubiquitin[:30]):
    ax.text(i, -1.5, aa, ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 3. BLOSUM62 Encoding

In [ ]:
# BLOSUM62 substitution matrix
BLOSUM62 = np.array([
    [ 4, 0,-2,-1,-2, 0,-2,-1,-1,-1,-1,-2,-1,-1,-1, 1, 0, 0,-3,-2],  # A
    [ 0, 9,-3,-4,-2,-3,-3,-1,-3,-1,-1,-3,-3,-3,-3,-1,-1,-1,-2,-2],  # C
    [-2,-3, 6, 2,-3,-1,-1,-3,-1,-4,-3, 1,-1, 0,-2, 0,-1,-3,-4,-3],  # D
    [-1,-4, 2, 5,-3,-2, 0,-3, 1,-3,-2, 0,-1, 2, 0, 0,-1,-2,-3,-2],  # E
    [-2,-2,-3,-3, 6,-3,-1, 0,-3, 0, 0,-3,-4,-3,-3,-2,-2,-1, 1, 3],  # F
    [ 0,-3,-1,-2,-3, 6,-2,-4,-2,-4,-3, 0,-2,-2,-2, 0,-2,-3,-2,-3],  # G
    [-2,-3,-1, 0,-1,-2, 8,-3,-1,-3,-2, 1,-2, 0, 0,-1,-2,-3,-2, 2],  # H
    [-1,-1,-3,-3, 0,-4,-3, 4,-3, 2, 1,-3,-3,-3,-3,-2,-1, 3,-3,-1],  # I
    [-1,-3,-1, 1,-3,-2,-1,-3, 5,-2,-1, 0,-1, 1, 2, 0,-1,-2,-3,-2],  # K
    [-1,-1,-4,-3, 0,-4,-3, 2,-2, 4, 2,-3,-3,-2,-2,-2,-1, 1,-2,-1],  # L
    [-1,-1,-3,-2, 0,-3,-2, 1,-1, 2, 5,-2,-2, 0,-1,-1,-1, 1,-1,-1],  # M
    [-2,-3, 1, 0,-3, 0, 1,-3, 0,-3,-2, 6,-2, 0, 0, 1, 0,-3,-4,-2],  # N
    [-1,-3,-1,-1,-4,-2,-2,-3,-1,-3,-2,-2, 7,-1,-2,-1,-1,-2,-4,-3],  # P
    [-1,-3, 0, 2,-3,-2, 0,-3, 1,-2, 0, 0,-1, 5, 1, 0,-1,-2,-2,-1],  # Q
    [-1,-3,-2, 0,-3,-2, 0,-3, 2,-2,-1, 0,-2, 1, 5,-1,-1,-3,-3,-2],  # R
    [ 1,-1, 0, 0,-2, 0,-1,-2, 0,-2,-1, 1,-1, 0,-1, 4, 1,-2,-3,-2],  # S
    [ 0,-1,-1,-1,-2,-2,-2,-1,-1,-1,-1, 0,-1,-1,-1, 1, 5, 0,-2,-2],  # T
    [ 0,-1,-3,-2,-1,-3,-3, 3,-2, 1, 1,-3,-2,-2,-3,-2, 0, 4,-3,-1],  # V
    [-3,-2,-4,-3, 1,-2,-2,-3,-3,-2,-1,-4,-4,-2,-3,-3,-2,-3,11, 2],  # W
    [-2,-2,-3,-2, 3,-3, 2,-1,-2,-1,-1,-2,-3,-1,-2,-2,-2,-1, 2, 7],  # Y
], dtype=np.float32)

print(f"BLOSUM62 shape: {BLOSUM62.shape}")

In [ ]:
# Visualize BLOSUM62 matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(BLOSUM62, cmap='RdBu_r', vmin=-4, vmax=11)
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(list(AMINO_ACIDS))
ax.set_yticklabels(list(AMINO_ACIDS))
ax.set_title('BLOSUM62 Substitution Matrix')
plt.colorbar(im, label='Log-odds score')
plt.tight_layout()
plt.show()

In [ ]:
def blosum_encode(sequence):
    """
    Encode sequence using BLOSUM62 rows.
    
    Each amino acid is represented by its substitution profile.
    """
    encoding = np.zeros((len(sequence), 20), dtype=np.float32)
    
    for i, aa in enumerate(sequence.upper()):
        if aa in AA_TO_IDX:
            encoding[i] = BLOSUM62[AA_TO_IDX[aa]]
    
    # Normalize to [-1, 1] range
    encoding = encoding / 9.0
    
    return encoding

ubq_blosum = blosum_encode(ubiquitin)
print(f"Shape: {ubq_blosum.shape}")
print(f"Value range: [{ubq_blosum.min():.2f}, {ubq_blosum.max():.2f}]")

In [ ]:
# Compare similar amino acids in BLOSUM space
# Compute pairwise similarities
similarities = BLOSUM62 @ BLOSUM62.T

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(similarities, cmap='viridis')
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(list(AMINO_ACIDS))
ax.set_yticklabels(list(AMINO_ACIDS))
ax.set_title('BLOSUM62 Encoding Similarity (dot product)')
plt.colorbar(im)
plt.show()

# Find most similar pairs
print("\nMost similar amino acid pairs (off-diagonal):")
triu_idx = np.triu_indices(20, k=1)
pair_sims = [(AMINO_ACIDS[i], AMINO_ACIDS[j], similarities[i,j]) 
             for i, j in zip(*triu_idx)]
for aa1, aa2, sim in sorted(pair_sims, key=lambda x: -x[2])[:5]:
    print(f"  {aa1}-{aa2}: {sim:.1f}")

## 4. Physicochemical Properties

In [ ]:
# Properties: hydrophobicity, volume, charge, polarity, aromaticity
PROPERTIES = {
    'A': [ 0.62, 0.11, 0.0, 0.0, 0.0],
    'C': [ 0.29, 0.24, 0.0, 0.0, 0.0],
    'D': [-0.90, 0.27,-1.0, 1.0, 0.0],
    'E': [-0.74, 0.40,-1.0, 1.0, 0.0],
    'F': [ 1.19, 0.55, 0.0, 0.0, 1.0],
    'G': [ 0.48, 0.00, 0.0, 0.0, 0.0],
    'H': [-0.40, 0.43, 0.5, 1.0, 1.0],
    'I': [ 1.38, 0.45, 0.0, 0.0, 0.0],
    'K': [-1.50, 0.53, 1.0, 1.0, 0.0],
    'L': [ 1.06, 0.45, 0.0, 0.0, 0.0],
    'M': [ 0.64, 0.47, 0.0, 0.0, 0.0],
    'N': [-0.78, 0.32, 0.0, 1.0, 0.0],
    'P': [ 0.12, 0.26, 0.0, 0.0, 0.0],
    'Q': [-0.85, 0.43, 0.0, 1.0, 0.0],
    'R': [-2.53, 0.60, 1.0, 1.0, 0.0],
    'S': [-0.18, 0.14, 0.0, 1.0, 0.0],
    'T': [-0.05, 0.26, 0.0, 1.0, 0.0],
    'V': [ 1.08, 0.33, 0.0, 0.0, 0.0],
    'W': [ 0.81, 0.74, 0.0, 0.0, 1.0],
    'Y': [ 0.26, 0.60, 0.0, 1.0, 1.0],
}

PROPERTY_NAMES = ['Hydrophobicity', 'Volume', 'Charge', 'Polarity', 'Aromaticity']

def physicochemical_encode(sequence):
    """Encode using 5 physicochemical properties."""
    encoding = np.zeros((len(sequence), 5), dtype=np.float32)
    for i, aa in enumerate(sequence.upper()):
        if aa in PROPERTIES:
            encoding[i] = PROPERTIES[aa]
    return encoding

ubq_physchem = physicochemical_encode(ubiquitin)
print(f"Shape: {ubq_physchem.shape}")

In [ ]:
# Visualize property encoding
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)

for i, (ax, name) in enumerate(zip(axes, PROPERTY_NAMES)):
    ax.bar(range(len(ubiquitin)), ubq_physchem[:, i], 
           color='steelblue' if i != 2 else 
           ['red' if v < 0 else 'blue' if v > 0 else 'gray' for v in ubq_physchem[:, i]])
    ax.set_ylabel(name)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

axes[-1].set_xlabel('Residue Position')
fig.suptitle('Physicochemical Properties of Ubiquitin', y=1.02)
plt.tight_layout()
plt.show()

## 5. Comparing Encodings

In [ ]:
# Compare dimensions
encodings = {
    'One-hot': ubq_onehot,
    'BLOSUM': ubq_blosum,
    'Physicochemical': ubq_physchem,
}

print("Encoding Comparison:")
print("-" * 50)
for name, enc in encodings.items():
    print(f"{name:20s} | Shape: {str(enc.shape):12s} | Total params: {enc.size}")

In [ ]:
# Analyze amino acid similarity in each encoding
from sklearn.metrics.pairwise import cosine_similarity

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, enc) in zip(axes, encodings.items()):
    # Get encoding for each amino acid type
    aa_encodings = {}
    for aa in AMINO_ACIDS:
        if name == 'One-hot':
            aa_encodings[aa] = one_hot_encode(aa, include_unknown=False)[0]
        elif name == 'BLOSUM':
            aa_encodings[aa] = blosum_encode(aa)[0]
        else:
            aa_encodings[aa] = physicochemical_encode(aa)[0]
    
    aa_matrix = np.array([aa_encodings[aa] for aa in AMINO_ACIDS])
    sim_matrix = cosine_similarity(aa_matrix)
    
    im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(20))
    ax.set_yticks(range(20))
    ax.set_xticklabels(list(AMINO_ACIDS), fontsize=8)
    ax.set_yticklabels(list(AMINO_ACIDS), fontsize=8)
    ax.set_title(f'{name} Similarity')

plt.colorbar(im, ax=axes, label='Cosine Similarity', shrink=0.8)
plt.tight_layout()
plt.show()

## 6. Combined Encoding

In [ ]:
def combined_encode(sequence):
    """
    Combine all encoding methods.
    
    Returns: (L, 45) array
        - One-hot: 20 dims
        - BLOSUM: 20 dims  
        - Physicochemical: 5 dims
    """
    onehot = one_hot_encode(sequence, include_unknown=False)
    blosum = blosum_encode(sequence)
    physchem = physicochemical_encode(sequence)
    
    return np.concatenate([onehot, blosum, physchem], axis=1)

ubq_combined = combined_encode(ubiquitin)
print(f"Combined encoding shape: {ubq_combined.shape}")

## Summary

| Encoding | Dimensions | Information Captured |
|----------|------------|---------------------|
| One-hot | 20 | Identity only |
| BLOSUM62 | 20 | Evolutionary substitution patterns |
| Physicochemical | 5 | Biochemical properties |
| Combined | 45 | All of the above |

**Key insights:**
- One-hot treats all amino acids as equally different
- BLOSUM captures evolutionary similarity (I ≈ L ≈ V)
- Physicochemical properties are interpretable and low-dimensional
- Modern approaches use learned embeddings (ESM, next lecture)